<a href="https://colab.research.google.com/github/SAfonso/paper-to-csv/blob/main/OCRTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Instalación de dependencias (Ejecutar en una celda separada en Colab)
!pip install easyocr opencv-python-headless pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 21.5 MB/s eta 0:00:00


In [10]:
import os
import cv2
import easyocr
import pandas as pd
import numpy as np
import re
from google.colab import drive

In [5]:
# Montar Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# 2. Rutas
RUTA_FOTOS = '/content/drive/MyDrive/PapelesOncle'
RUTA_SALIDA_CSV = '/content/drive/MyDrive/PapelesOncle/contactos_final.csv'

In [7]:
lista_contactos = []

# Inicializar el lector de OCR (descargará los modelos la primera vez)
print("Cargando modelo OCR...")
reader = easyocr.Reader(['es', 'en'], gpu=True) # Cambia gpu=False si no usas entorno T4 en Colab

Cargando modelo OCR...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [17]:
# 3. Función de extracción y validación lógica
def extraer_datos_por_geometria(resultados_ocr):
    # Debug temporal — bórralo cuando funcione
    if resultados_ocr:
        print(f"  [DEBUG] Ejemplo bbox: {resultados_ocr[0][0]}, texto: '{resultados_ocr[0][1]}'")
    else:
        print("  [DEBUG] resultados_ocr está vacío")
        return "No detectado", "No detectado"


    nombre_detectado = ""
    email_detectado = []

    y_nombre_ancla = -1
    y_email_ancla = -1
    y_suelo_legal = float('inf')

    ETIQUETAS_IGNORAR = {
        "sorteo", "2", "entradas", "nombre", "nombre:",
        "email", "email:", "medianamente", "comedy"
    }

    # --- MEJORA 1: Ordenar por Y antes de procesar ---
    # Garantiza procesamiento top-to-bottom, más predecible
    resultados_ordenados = sorted(resultados_ocr, key=lambda x: (x[0][0][1] + x[0][2][1]) / 2)

    # Primera pasada: encontrar anclas
    for (bbox, texto, prob) in resultados_ordenados:
        texto_limpio = texto.lower().replace(" ", "")
        y_centro = (bbox[0][1] + bbox[2][1]) / 2

        if "nombre" in texto_limpio and y_nombre_ancla == -1:
            y_nombre_ancla = y_centro

        # --- MEJORA 2: Ancla explícita para el email ---
        elif "email" in texto_limpio and y_email_ancla == -1:
            y_email_ancla = y_centro

        elif "medianamente" in texto_limpio:
            if y_centro < y_suelo_legal:
                y_suelo_legal = y_centro

    # Fallback: si no se detectó suelo legal, usar 80% de la imagen
    if y_suelo_legal == float('inf') and resultados_ordenados:
      try:
        # EasyOCR devuelve bbox como lista de 4 puntos: [[x0,y0],[x1,y1],[x2,y2],[x3,y3]]
        # El punto [2] es bottom-right, [0] es top-left
        ys = []
        for (bbox, texto, prob) in resultados_ordenados:
          if len(bbox) >= 3:
            ys.append(bbox[2][1])  # y del punto bottom-right
        y_suelo_legal = max(ys) * 0.75 if ys else 9999
      except Exception as e:
        print(f"  [WARN] Fallback y_suelo_legal falló: {e}")
        y_suelo_legal = 9999

    # Segunda pasada: clasificar
    for (bbox, texto, prob) in resultados_ordenados:
        y_centro = (bbox[0][1] + bbox[2][1]) / 2
        texto_limpio = texto.strip()

        # --- MEJORA 3: Filtro de etiquetas más robusto ---
        if texto_limpio.lower().replace(":", "").replace(" ", "") in ETIQUETAS_IGNORAR:
            continue

        # Zona NOMBRE: misma línea que la etiqueta "NOMBRE:" con tolerancia ampliada
        # --- MEJORA 4: Tolerancia aumentada a 40px ---
        if y_nombre_ancla != -1 and abs(y_centro - y_nombre_ancla) < 40:
            nombre_detectado += texto_limpio + " "

        # Zona EMAIL: entre la línea del email y el texto legal
        # --- MEJORA 5: Usar ancla de email si existe, sino usar zona entre nombre y legal ---
        elif y_email_ancla != -1 and y_email_ancla < y_centro < y_suelo_legal:
            email_detectado.append(texto_limpio)
        elif y_email_ancla == -1 and y_nombre_ancla < y_centro < y_suelo_legal:
            email_detectado.append(texto_limpio)

    nombre_final = nombre_detectado.strip() if nombre_detectado else "No detectado"

    # --- MEJORA 6: Limpieza de email más agresiva ---
    if email_detectado:
        email_raw = "".join(email_detectado)
        # Quitar espacios y caracteres raros comunes en OCR
        email_raw = email_raw.replace(" ", "").replace("\n", "")
        # Correcciones OCR frecuentes en emails
        email_raw = email_raw.replace("@gmai1", "@gmail").replace("@hotmai1", "@hotmail")
        # Validar que parece un email
        email_final = email_raw if re.search(r'@\w+\.\w+', email_raw) else f"REVISAR: {email_raw}"
    else:
        email_final = "No detectado"

    return nombre_final, email_final



In [14]:
def preprocesar_para_ocr(img):  # <-- recibe el array, no el path
    # Quitar el cv2.imread, ya viene leída
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    binary = cv2.adaptiveThreshold(
        enhanced, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    return binary

In [18]:
# 4. Bucle principal
print("Procesando imágenes...")
for archivo in os.listdir(RUTA_FOTOS):
    if archivo.lower().endswith(('.jpg', '.jpeg', '.png')):
        ruta_imagen = os.path.join(RUTA_FOTOS, archivo)

        # Leer imagen
        img = cv2.imread(ruta_imagen)
        if img is None:
            continue

        # Primero procesamos la imagen
        img_procesada = preprocesar_para_ocr(img)

        # Pasar OCR a toda la imagen
        resultados = reader.readtext(img_procesada)

        # Extraer lógica
        nombre, email = extraer_datos_por_geometria(resultados)

        lista_contactos.append([nombre, email])
        print(f"[{archivo}] -> Nombre: {nombre} | Email: {email}")

Procesando imágenes...
  [DEBUG] Ejemplo bbox: [[np.int32(332), np.int32(810)], [np.int32(1159), np.int32(810)], [np.int32(1159), np.int32(930)], [np.int32(332), np.int32(930)]], texto: 'RIE@2ENIRADAS'
[WhatsApp Image 2026-06-08 at 22.03.43.jpeg] -> Nombre: No detectado | Email: REVISAR: RIE@2ENIRADASNGIMIBREEndi
  [DEBUG] Ejemplo bbox: [[np.int32(467), np.int32(913)], [np.int32(567), np.int32(913)], [np.int32(567), np.int32(1001)], [np.int32(467), np.int32(1001)]], texto: 'MeC'
[WhatsApp Image 2026-06-08 at 23.44.06.jpeg] -> Nombre: No detectado | Email: REVISAR: MeC@24NPADMBRE}Me
  [DEBUG] Ejemplo bbox: [[np.int32(1097), np.int32(957)], [np.int32(1145), np.int32(957)], [np.int32(1145), np.int32(977)], [np.int32(1097), np.int32(977)]], texto: '1a4'
[WhatsApp Image 2026-06-08 at 23.44.07.jpeg] -> Nombre: No detectado | Email: REVISAR: 5141eney2*08421a4f@5uxy
  [DEBUG] Ejemplo bbox: [[np.int32(311), np.int32(794)], [np.int32(505), np.int32(794)], [np.int32(505), np.int32(905)], [np.int3